# DDS + CIC Calibration and Linearity Analysis

This notebook explains how to measure, analyze, and calibrate the output of an ADC followed by a Polyphase Filter Bank (PFB) and a DDS + CIC block. 

## 1. System Overview

We consider the following chain:

1. RF signal entering an ADC:
   - Frequency: $f_{RF} = 300\,\mathrm{MHz}$
   - Power: variable $P_{in}$ (dBm)
   - ADC sampling rate: $F_s = 4096\,\mathrm{MHz}$
2. Digital decimation by 2 and mixing by $-F_s/4$ to create a complex baseband.
3. Polyphase Filter Bank (PFB) with 8 channels and decimation factor 8:
   - Output bandwidth per channel: 256 MHz
   - PFB output scaling controlled by `qout(N)`
4. DDS to translate PFB output signal to DC
5. CIC for decimating the signal
   - Decimation factor R = 8
   - Differential delay M = 1
   - Order N=3
   - DDS + CIC block output scaling controlled by `qprod(14)`

The goal is to measure the tone level at the output of the DDS + CIC and convert it to real RF power in dBm.

---

## Gain Compensation Between PFB and CIC

In the processing chain

```
PFB → DDS → CIC
```

the CIC decimator introduces a significant gain that must be taken into account to avoid overflow and to maintain a convenient signal scaling.

For a CIC filter with:

* Decimation factor (R = 8)
* Differential delay (M = 1)
* Order (N = 3)

the DC gain of the filter is

$$
G_{CIC} = (RM)^N
$$

which gives

$$
G_{CIC} = (8 \cdot 1)^3 = 512
$$

This means that the CIC increases the signal amplitude by a factor of 512 at low frequencies.

---

### Compensation Using PFB Output Scaling

The PFB output stage includes a configurable scaling parameter `QOUT_REG`, which effectively divides the output amplitude by

$$
2^{QOUT\_REG}
$$

If $QOUT\_REG = 9$ then

$$
G_{PFB} = 2^{-9} = \frac{1}{512}
$$

Therefore the combined gain becomes

$$
G_{PFB} \cdot G_{CIC} \approx 1
$$

which keeps the signal amplitude approximately constant across the PFB and CIC stages.

---

### Practical Implication

Selecting

```
QOUT_REG = 9
```

provides a convenient scaling where the CIC gain is approximately compensated by the PFB output scaling, helping to prevent overflow while maintaining a consistent signal level throughout the DSP chain.

---

## Quantization Scaling at the CIC Output

After the CIC filter, the signal is reduced from 24 bits to 16 bits using the `qdata` module.

The module selects a 16-bit slice of the input word according to

```verilog
qsel_i = BIN - QSEL_REG - 1
dout   = din[qsel_i -: BOUT]
```

For the configuration used in this experiment:

```
BIN  = 24
BOUT = 16
QSEL_REG = 14
```

This effectively corresponds to a right shift of the input signal by $QSEL\_REG$ bits, resulting in an amplitude scaling

$$
G_{QDATA} = 2^{-QSEL\_REG}
$$

Thus, for $QSEL\_REG = 14$ the gain is

$$
G_{QDATA} = 2^{-14} = \frac{1}{16384}
$$

---

## Total Gain of the DSP Chain

The signal processing chain is:

```
PFB → DDS → CIC → qdata
```

Each stage contributes a gain factor.

### PFB Scaling

The PFB output includes a configurable scaling factor.
For this configuration:

$$
G_{PFB} = 2^{-6}
$$

---

### CIC Gain

For a CIC decimator with

* Decimation factor (R = 8)
* Differential delay (M = 1)
* Order (N = 3)

the DC gain is

$$
G_{CIC} = (RM)^N
$$

which gives

$$
G_{CIC} = (8 \cdot 1)^3 = 512
$$

---

### Quantization Scaling

The `qdata` module introduces an additional scaling

$$
G_{QDATA} = 2^{-14}
$$

---

## Combined Gain

Combining the gains of all stages:

$$
G_{TOTAL} =
G_{PFB} \cdot G_{CIC} \cdot G_{QDATA}
$$

$$
G_{TOTAL} =
2^{-6} \times 512 \times 2^{-14}
$$

Since

$$
512 = 2^9
$$

we obtain

$$
G_{TOTAL} =
2^{-6} \times 2^9 \times 2^{-14}
$$

$$
G_{TOTAL} = 2^{-11}
$$

---

## Final Result

The complete DSP chain introduces an overall amplitude scaling of

$$
G_{TOTAL} = 2^{-11}
$$

which means that the final 16-bit output corresponds to the internal signal scaled by

$$
\frac{1}{2048}
$$

before being written to the output stream.

---

## Gain Summary of the DSP Chain

The following table summarizes the gain introduced by each stage of the processing chain:

| Stage         | Gain (linear) | Gain (dB)     | Description                                         |
| ------------- | ------------- | ------------- | --------------------------------------------------- |
| PFB scaling   | ($2^{-6}$)      | −36.12 dB     | Output scaling applied in the PFB block             |
| CIC filter    | ($512 = 2^9$)   | +54.19 dB     | Gain of the CIC decimator (R=8, M=1, N=3)           |
| qdata scaling | ($2^{-14}$)     | −84.29 dB     | Bit selection / quantization scaling (24 → 16 bits) |
| **Total**     | ($2^{-11}$)     | **−66.22 dB** | Overall gain of the DSP chain                       |

---

## Interpretation

Combining the gains of all stages:

$$
G_{TOTAL} =
2^{-6} \cdot 2^{9} \cdot 2^{-14}
$$

$$
G_{TOTAL} = 2^{-11}
$$

which corresponds to

$$
G_{TOTAL} = \frac{1}{2048}
$$

or

$$
-66.22 \text{ dB}
$$

Therefore, the signal amplitude at the final 16-bit output is **2048 times smaller** than the internal signal level before these scaling stages.

This known scaling factor is useful when relating the digital amplitude measured in the output data to the **actual signal power at the ADC input**.

---


# DDS+CIC into the signal chain

---

### **1. Decimation Handling**

```python
D = 8
if D == 1:
    chain.analysis.set_ddscic_outsel(data="product", cic="no")
else:
    chain.analysis.set_ddscic_outsel(data="product", cic="yes")
    chain.analysis.set_ddscic_decimation(D)
```

* If `D == 1`, the CIC is bypassed, as DDS+CIC requires decimation ≥ 2 to operate.
* For `D > 1`, the CIC is enabled, and the decimation factor is set.

**Note:** CIC filters introduce **amplitude droop** and **phase distortion** depending on decimation factor and number of stages. Keep this in mind when analyzing post-DDC signals.

---

### **2. DDS Frequency**

```python
f_ddscic = fout - FC
chain.analysis.set_ddscic_ddsfreq(f=f_ddscic*1e6)
```

* This sets the DDS frequency relative to your PFB output channel center.
* Make sure the `fout` is within the Nyquist range after decimation: ($ f_{out} < \frac{f_s}{2} $).

---

### **3. Quantization**

```python
chain.analysis.set_ddscic_qprod(14)  # 0-16 bits
```

* This sets the quantization of the output.
* Lower bit width increases quantization noise but reduces resource usage.

---

### **4. Sampling frequency after decimation**

```python
fs_d = chain.fs_ch / chain.analysis.get_ddscic_decimation()
```

* This is the **effective sampling rate** of the post-DDC signal.
* Use `fs_d` for FFT and phase calculations.

---

### **5. Time-domain signal**

```python
[xi,xq] = chain.get_bin_pfb(fout, verbose=True)
x_postddscic_t = xi + 1j*xq
```

* `xi` and `xq` are the in-phase and quadrature components.
* `x_postddscic_t` is the **complex baseband signal** after DDS+CIC.

---

### **6. Saving the data**

```python
np.savetxt(QICK_TOOLS + "/.../x_postddscic_t_data.txt", x_postddscic_t)
```

* Now we have a full time-domain record for later spectrum, amplitude, or phase analysis.

---

### **Next Steps for Analysis**

1. **Amplitude spectrum**

```python
X = np.fft.fftshift(np.fft.fft(x_postddscic_t))
f_axis = np.fft.fftshift(np.fft.fftfreq(len(X), 1/fs_d))
plt.plot(f_axis/1e6, 20*np.log10(np.abs(X)))
plt.xlabel('Frequency [MHz]')
plt.ylabel('Amplitude [dB]')
plt.show()
```

2. **Phase response**

```python
plt.plot(f_axis/1e6, np.angle(X))
plt.xlabel('Frequency [MHz]')
plt.ylabel('Phase [rad]')
plt.show()
```

3. **Check CIC effect**

* If you decimate by `D`, CIC introduces a sinc-shaped amplitude response:
  $$
  H(f) = \left|\frac{\sin(\pi f D / f_s)}{D \sin(\pi f / f_s)}\right|^N
  $$
  where ($N$) = number of CIC stages.

4. **Compare pre- and post-DDC signals** to quantify amplitude droop and phase shift.

---

In [ ]:
%matplotlib ipympl
import numpy as np
import matplotlib.pyplot as plt

# ================================
# User parameters (update these)
# ================================
data_file = "data/f_300-10dBm/x_postddscic_t_data.txt"  # path to your saved data
fs_ch = 256e6       # ADC sampling frequency before decimation (Hz)
D = 8               # CIC decimation factor
N = 3               # Number of CIC stages (3rd order CIC as IP block)
# ================================

# Load complex time-domain data
x_postddscic_t = np.loadtxt(data_file, dtype=complex)

# Effective sampling rate after decimation
fs_d = fs_ch / D
print(f"Effective sampling rate after decimation: {fs_d/1e6:.2f} MHz")

# FFT
X = np.fft.fftshift(np.fft.fft(x_postddscic_t))
f_axis = np.fft.fftshift(np.fft.fftfreq(len(X), 1/fs_d))

# Amplitude in dB
amp_dB = 20*np.log10(np.abs(X)/np.max(np.abs(X)))

# Phase
phase_rad = np.angle(X)

# CIC theoretical amplitude response (normalized)
H_cic = np.sinc(f_axis * D / fs_ch)**N
H_cic_dB = 20*np.log10(H_cic/np.max(H_cic))

# -------------------------------
# Plot amplitude
plt.figure(figsize=(10,5))
plt.plot(f_axis/1e6, amp_dB, label='Measured post-DDSCIC')
plt.plot(f_axis/1e6, H_cic_dB, 'r--', label='CIC theoretical')
plt.xlabel('Frequency [MHz]')
plt.ylabel('Amplitude [dB]')
plt.title(f'Amplitude Spectrum (D={D}, N={N})')
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.show()

# -------------------------------
# Plot phase
plt.figure(figsize=(10,5))
plt.plot(f_axis/1e6, phase_rad)
plt.xlabel('Frequency [MHz]')
plt.ylabel('Phase [rad]')
plt.title('Phase Spectrum')
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# ================================
# User parameters
# ================================
data_pre = "data/f_300-10dBm/x_preddscic_t_data.txt"   # signal before DDS+CIC
data_post = "data/f_300-10dBm/x_postddscic_t_data.txt" # signal after DDS+CIC
fs_ch = 256e6       # ADC sampling frequency before decimation
D = 8               # CIC decimation factor
N = 3               # CIC order
# ================================

# Load signals
x_pre = np.loadtxt(data_pre, dtype=complex)
x_post = np.loadtxt(data_post, dtype=complex)

# Effective sampling rate after decimation
fs_d = fs_ch / D
print(f"Effective sampling rate after decimation: {fs_d/1e6:.2f} MHz")

# --- FFT for 'Before' (Full Rate) ---
X_pre = np.fft.fftshift(np.fft.fft(x_pre))
# Use the full ADC sampling rate here
f_axis_pre = np.fft.fftshift(np.fft.fftfreq(len(x_pre), 1/fs_ch))

# --- FFT for 'After' (Decimated Rate) ---
X_post = np.fft.fftshift(np.fft.fft(x_post))
# Use the decimated sampling rate here
f_axis_post = np.fft.fftshift(np.fft.fftfreq(len(x_post), 1/fs_d))

# Normalize amplitudes for comparison
amp_pre_dB = 20*np.log10(np.abs(X_pre)/np.max(np.abs(X_pre)))
amp_post_dB = 20*np.log10(np.abs(X_post)/np.max(np.abs(X_post)))

# Phase
phase_pre_rad = np.angle(X_pre)
phase_post_rad = np.angle(X_post)

# CIC theoretical response
H_cic = np.sinc(f_axis * D / fs_ch)**N
H_cic_dB = 20*np.log10(H_cic/np.max(H_cic))

# -------------------------------
# Plot amplitude comparison
plt.figure(figsize=(10,5))
# Plot 'Before' against the 256MHz-based axis
plt.plot(f_axis_pre/1e6, amp_pre_dB, label='Before DDS+CIC (Full BW)', alpha=0.7)
# Plot 'After' against the 32MHz-based axis
plt.plot(f_axis_post/1e6, amp_post_dB, label='After DDS+CIC (Decimated)', linewidth=2)
#plt.plot(f_axis_pre/1e6, amp_pre_dB, label='Before DDS+CIC')
#plt.plot(f_axis_post/1e6, amp_post_dB, label='After DDS+CIC')
plt.plot(f_axis/1e6, H_cic_dB, 'r--', label='3rd-order CIC theoretical')
plt.xlabel('Frequency [MHz]')
plt.ylabel('Amplitude [dB]')
plt.title('Amplitude Spectrum Comparison')
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.show()

# -------------------------------
# Plot phase comparison
plt.figure(figsize=(10,5))
plt.plot(f_axis_pre/1e6, phase_pre_rad, label='Before DDS+CIC')
plt.plot(f_axis_post/1e6, phase_post_rad, label='After DDS+CIC')
plt.xlabel('Frequency [MHz]')
plt.ylabel('Phase [rad]')
plt.title('Phase Spectrum Comparison')
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# ===============================
# LOAD DATA
# ===============================

pre  = np.loadtxt("data/f_300-10dBm/x_preddscic_t_data.txt", dtype=np.complex128)
post = np.loadtxt("data/f_300-10dBm/x_postddscic_t_data.txt", dtype=np.complex128)

Fs_pre  = 256e6
Fs_post = 32e6

# if complex IQ stored as I,Q
#if pre.ndim > 1:
#    pre = pre[:,0] + 1j*pre[:,1]

#if post.ndim > 1:
#    post = post[:,0] + 1j*post[:,1]

# ===============================
# TIME DOMAIN
# ===============================

plt.figure(figsize=(10,4))
plt.plot(np.real(pre[:2000000]), label="pre")
plt.plot(np.real(post[:2000000]), label="post")
plt.legend()
plt.title("Time domain comparison")
plt.show()

# ===============================
# FFT PRE
# ===============================

N = len(pre)

fft_pre = np.fft.fftshift(np.fft.fft(pre))
f_pre = np.fft.fftshift(np.fft.fftfreq(N,1/Fs_pre))

plt.figure(figsize=(7,5))
plt.plot(f_pre/1e6,20*np.log10(np.abs(fft_pre)))
plt.title("FFT before DDS+CIC")
plt.xlabel("MHz")
plt.ylabel("dB")
plt.show()

# ===============================
# FFT POST
# ===============================

N = len(post)

fft_post = np.fft.fftshift(np.fft.fft(post))
f_post = np.fft.fftshift(np.fft.fftfreq(N,1/Fs_post))

plt.figure(figsize=(7,5))
plt.plot(f_post/1e6,20*np.log10(np.abs(fft_post)))
plt.title("FFT after DDS+CIC")
plt.xlabel("MHz")
plt.ylabel("dB")
plt.show()

# ===============================
# AMPLITUDE
# ===============================

amp_pre  = np.abs(pre)
amp_post = np.abs(post)

print("Mean amplitude pre :", np.mean(amp_pre))
print("Mean amplitude post:", np.mean(amp_post))

# ===============================
# PHASE
# ===============================

phase_pre  = np.unwrap(np.angle(pre))
phase_post = np.unwrap(np.angle(post))

plt.figure(figsize=(8,4))
#plt.plot(phase_pre[:10000], label="pre")
plt.plot(phase_post[:60000], label="post")
plt.legend()
plt.title("Phase evolution")
plt.show()

gain_factor = np.mean(np.abs(x_post)) / np.mean(np.abs(x_pre))
print(f"Block gain: {gain_factor:.2f}")

# Tomamos la diferencia de fase entre el último y el primer punto
delta_phase = phase_post[-1] - phase_post[0]
delta_time = len(phase_post) / fs_d

# Calculamos el error de frecuencia en Hz
f_error = delta_phase / (2 * np.pi * delta_time)
print(f"Residual frequency error: {f_error:.2f} Hz")

## ADC Calibration with PFB and DDS+CIC Chain

### Complete Signal Chain
```
Pin_gen → [Analog Atten] → ADC → PFB → DDS → CIC → Output
```

**Chain breakdown:**
1. **Analog attenuation:** -12.89 dB
2. **ADC:** Converts to digital (characterized by $K_{\text{adc}}$)
3. **PFB:** Divides by $2^{q_{\text{out}}}$ where $q_{\text{out}} = 6$ (i.e., ÷64 = -36.12 dB)
4. **DDS:** Frequency translation (no gain/loss in ideal case)
5. **CIC:** Decimation filter with gain $G_{\text{CIC}} = (RM)^N$
6. **CIC:** Output scaling by $2^{q_{\text{prod}}}$ where $q_{\text{prod}} = 14$ (i.e., ÷16384 = -84.29 dB)

---

## CIC Filter Characteristics

### CIC Gain

For a CIC filter with:
- Decimation factor: $R = 8$
- Differential delay: $M = 1$
- Order: $N = 3$

The CIC gain is:
$$G_{\text{CIC}} = (RM)^N = (8 \times 1)^3 = 512$$

In decibels:
$$G_{\text{CIC,dB}} = 20 \log_{10}(512) = 54.19 \text{ dB}$$

Therefore, the **combined FFT+CIC gain is unity** (0 dB).

---

## Updated Calibration Equation

### Without FFT Pre-compensation

If the FFT does **not** compensate the CIC gain (i.e., `QOUT_REG ≠ 9`):

The complete chain has an additional gain/loss term from FFT and CIC:
$$\text{FFT\_CIC\_NET} = G_{\text{FFT,dB}} + G_{\text{CIC,dB}}$$

Where:
- $G_{\text{FFT,dB}} = -20 \log_{10}(2^{\text{QOUT\_REG}})$
- $G_{\text{CIC,dB}} = 20 \log_{10}(R^N) = 54.19$ dB (for $R=8, N=3$)

The calibration becomes:
$$\boxed{P_{\text{in,gen}} = P_{\text{output,digital}} + \text{CAL\_CONSTANT} - \text{FFT\_CIC\_NET}}$$

### With FFT Pre-compensation (QOUT_REG = 9)

If `QOUT_REG = 9`, then FFT_CIC_NET = 0 dB, and:
$$\boxed{P_{\text{in,gen}} = P_{\text{output,digital}} + \text{CAL\_CONSTANT}}$$

where `CAL_CONSTANT` remains:
$$\text{CAL\_CONSTANT} = \text{PFB\_SCALING} + \text{ANALOG\_ATTEN} - K_{\text{adc}}$$

---

## Effect on Signal Phase

### What QOUT_REG Does NOT Affect

The FFT quantization (`QOUT_REG`) **does not introduce deterministic phase rotation**. The signal phase is preserved.

### What CAN Affect Phase

1. **DDS:** Introduces phase shift $\phi = -2\pi f_{\text{DDS}} t$
   - This is a **linear phase** (frequency translation)
   - Does not change the phase of the tone relative to DC after translation
   
2. **CIC Filter:** Introduces **group delay**
   $$\tau_g = \frac{N(R-1)}{2}$$
   For $N=3, R=8$: $\tau_g = 10.5$ samples
   - This is a **constant delay** (linear phase in frequency)
   - Does not affect relative phase measurements of a single tone

3. **Quantization noise:** Adds small random phase error for low SNR

### Practical Implication

For a single tone:
- **Amplitude** is affected by `QOUT_REG`, CIC gain, etc.
- **Phase** is essentially unchanged (except for predictable linear phase from DDS and CIC delay)
- Phase measurements are **consistent across power levels** if system is linear

---

## Code: Complete Calibration with DDS+CIC

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import glob, re

# ==========================================
# SYSTEM PARAMETERS
# ==========================================
fs_pfb = 256e6      # PFB output rate
fs_adc = 2048e6     # ADC input rate
ANALOG_ATTEN = 12.89  # dB analog attenuation

# PFB
PFB_CHANNELS = 8
PFB_QOUT = 6
PFB_SCALING = 20*np.log10(2**PFB_QOUT)  # dB scaling due to PFB output

# CIC
D = 8   # Decimation
N = 3   # Order
R = D
M = 1
CIC_GAIN = (R*M)**N
CIC_GAIN_DB = 20*np.log10(CIC_GAIN)
fs_out = fs_pfb / D

# DDS+CIC configuration for this measurement
QPROD = 14               # The configured DDS product quantization used in this measurement
QPROD_DDS_INTERNAL = 24  # Internal DDS product width
CIC_OUTPUT_BITS = 16      # Actual output bits

# Pre-DDS calibration
CAL_CONSTANT_PRE = 30.33
K_ADC = PFB_SCALING + ANALOG_ATTEN - CAL_CONSTANT_PRE

# ==========================================
# LOAD DATA
# ==========================================
data_path = "data/f_300*/x_postddscic_t_data.txt"
files = sorted(glob.glob(data_path), key=lambda f: float(re.search(r'(-?\+?\d+)dBm', f).group(1)))

FS = 2**15
tone_bins = 3
measurements = []

# ==========================================
# PROCESS FILES
# ==========================================
for file in files:
    Pin_gen = float(re.search(r'(-?\+?\d+)dBm', file).group(1))
    x = np.loadtxt(file, dtype=np.complex128)
    N_samples = len(x)
    max_val = np.max(np.abs(x))
    std_val = np.std(np.abs(x))
    
    # Window & FFT
    window = np.hanning(N_samples)
    coherent_gain = np.sum(window)/N_samples
    xw = x*window
    fft_complex = np.fft.fftshift(np.fft.fft(xw)/N_samples)
    freqs = np.fft.fftshift(np.fft.fftfreq(N_samples, 1/fs_out))
    mag = np.abs(fft_complex)/coherent_gain
    
    # Peak detection & tone power
    peak_idx = np.argmax(mag)
    f_peak = freqs[peak_idx]
    idx0, idx1 = peak_idx - tone_bins, peak_idx + tone_bins + 1
    tone_power = np.sum(mag[idx0:idx1]**2)
    P_output_dbfs = 10*np.log10(tone_power/(FS**2))
    
    # Phase
    tone_phase = np.angle(fft_complex[peak_idx])
    
    # SNR & SFDR
    noise_mask = np.ones(len(mag), dtype=bool)
    noise_mask[idx0:idx1] = False
    noise_power = np.mean(mag[noise_mask]**2)
    snr_db = 10*np.log10(tone_power/(len(mag)*noise_power))
    mag_no_tone = mag.copy(); mag_no_tone[idx0:idx1] = 0
    sfdr_db = 10*np.log10(tone_power/np.max(mag_no_tone)**2)
    
    measurements.append({
        'Pin_gen': Pin_gen,
        'P_output_dbfs': P_output_dbfs,
        'f_peak': f_peak,
        'phase': tone_phase,
        'max_val': max_val,
        'std_val': std_val,
        'snr_db': snr_db,
        'sfdr_db': sfdr_db,
        'file': file
    })
    
    saturation_pct = (max_val/FS)*100
    sat_flag = " ⚠️ SAT" if saturation_pct > 90 else ""
    print(f"Pin={Pin_gen:+5.0f} dBm → Out={P_output_dbfs:7.2f} dBFS, Phase={np.degrees(tone_phase):+7.1f}°, SNR={snr_db:5.1f} dB, Peak={saturation_pct:4.1f}%{sat_flag}")

# ==========================================
# CALIBRATION
# ==========================================
linear_meas = measurements[:8]
Pin_vals = np.array([m['Pin_gen'] for m in linear_meas])
Pout_vals = np.array([m['P_output_dbfs'] for m in linear_meas])
coeffs = np.polyfit(Pout_vals, Pin_vals, 1)
slope = coeffs[0]
CAL_CONSTANT_POST = coeffs[1]

print(f"\nLinear fit: Pin = {slope:.3f} × P_out + {CAL_CONSTANT_POST:.2f} dB")

# ==========================================
# DDS+CIC QUANTIZATION AND GAIN ANALYSIS
# ==========================================
print(f"\n{'='*80}")
print("DDS+CIC QUANTIZATION AND GAIN ANALYSIS")
print(f"{'='*80}")

TRUNC_BITS = QPROD_DDS_INTERNAL - CIC_OUTPUT_BITS
TRUNC_DB = 6.02*TRUNC_BITS
DDS_CIC_NET_EFFECT = CAL_CONSTANT_PRE - CAL_CONSTANT_POST
UNACCOUNTED_LOSS = CIC_GAIN_DB - DDS_CIC_NET_EFFECT
residual_loss = UNACCOUNTED_LOSS - TRUNC_DB

print(f"Configured QPROD = {QPROD} (used for this data)")
print(f"Internal DDS width: {QPROD_DDS_INTERNAL} bits, CIC output: {CIC_OUTPUT_BITS} bits")
print(f"Truncation effect: {TRUNC_BITS} bits → {TRUNC_DB:.2f} dB")
print(f"Theoretical CIC gain: {CIC_GAIN_DB:.2f} dB")
print(f"Measured net gain: {DDS_CIC_NET_EFFECT:.2f} dB (Theoretical CIC gain - Truncation effect: {CIC_GAIN_DB-TRUNC_DB:.2f} dB)")
print(f"Model mismatch: {residual_loss:.2f} dB (likely internal scaling/truncation effects)")

# ==========================================
# PHASE ANALYSIS
# ==========================================
Pin_list = [m['Pin_gen'] for m in measurements]
phases_deg = [np.degrees(m['phase']) for m in measurements]
phase_mean = np.mean(phases_deg)
phase_std = np.std(phases_deg)
phase_power_corr = np.corrcoef(Pin_list, phases_deg)[0,1]

print(f"\nPhase mean: {phase_mean:.2f}°, std dev: {phase_std:.2f}°")
print(f"Phase-Power correlation: {phase_power_corr:.3f}")

# ==========================================
# PLOTTING
# ==========================================
fig = plt.figure(figsize=(18, 14))
gs = fig.add_gridspec(4, 3, hspace=0.35, wspace=0.35)

# Calibration
ax1 = fig.add_subplot(gs[0, 0])
Pin_recovered = [m['P_output_dbfs'] + CAL_CONSTANT_POST for m in measurements]
errors = np.array(Pin_recovered) - np.array(Pin_list)
ax1.plot(Pin_list, Pin_recovered, 'o-', ms=8, lw=2, color='blue', label='Recovered')
ax1.plot(Pin_list, Pin_list, 'k--', lw=2, label='Ideal')
ax1.set_xlabel('True Input Power (dBm)', fontsize=11)
ax1.set_ylabel('Recovered Power (dBm)', fontsize=11)
ax1.set_title('Post-DDS+CIC Calibration', fontsize=12, fontweight='bold')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Calibration Error
ax2 = fig.add_subplot(gs[0, 1])
ax2.plot(Pin_list, errors, 'o-', ms=8, lw=2, color='green')
ax2.axhline(0, color='k', ls='--', lw=2)
ax2.fill_between(Pin_list, -1, 1, alpha=0.15, color='green')
ax2.set_xlabel('Input Power (dBm)', fontsize=11)
ax2.set_ylabel('Error (dB)', fontsize=11)
ax2.set_title(f'Error (σ={np.std(errors[:8]):.3f} dB)', fontsize=12, fontweight='bold')
ax2.grid(True, alpha=0.3)

# Phase vs Power (KEY DIAGNOSTIC)
ax3 = fig.add_subplot(gs[0, 2])
snr_list = [m['snr_db'] for m in measurements]
scatter = ax3.scatter(Pin_list, phases_deg, c=snr_list, s=100, 
                     cmap='viridis', edgecolors='k', linewidth=1)
ax3.axhline(phase_mean, color='r', ls='--', lw=2, alpha=0.7)
if abs(phase_power_corr) > 0.5:
    coeffs_phase = np.polyfit(Pin_list, phases_deg, 1)
    p_fit = np.poly1d(coeffs_phase)
    ax3.plot(Pin_list, p_fit(Pin_list), 'r-', lw=2, alpha=0.5,
             label=f'r={phase_power_corr:.2f}')
    ax3.legend()
ax3.set_xlabel('Input Power (dBm)', fontsize=11)
ax3.set_ylabel('Phase (degrees)', fontsize=11)
ax3.set_title(f'Phase vs Power (σ={phase_std:.1f}°)', fontsize=12, fontweight='bold')
plt.colorbar(scatter, ax=ax3, label='SNR (dB)')
ax3.grid(True, alpha=0.3)

# Saturation Check
ax4 = fig.add_subplot(gs[1, 0])
max_vals = [m['max_val'] for m in measurements]
ax4.plot(Pin_list, max_vals, 'o-', ms=8, lw=2, color='red')
ax4.axhline(FS, color='k', ls='-', lw=2, label='Full Scale')
ax4.axhline(0.9*FS, color='orange', ls='--', lw=2, label='90% FS')
ax4.set_xlabel('Input Power (dBm)', fontsize=11)
ax4.set_ylabel('Peak Amplitude', fontsize=11)
ax4.set_title('Saturation Check', fontsize=12, fontweight='bold')
ax4.legend()
ax4.grid(True, alpha=0.3)

# SNR
ax5 = fig.add_subplot(gs[1, 1])
ax5.plot(Pin_list, snr_list, 'o-', ms=8, lw=2, color='green')
ax5.set_xlabel('Input Power (dBm)', fontsize=11)
ax5.set_ylabel('SNR (dB)', fontsize=11)
ax5.set_title('Signal-to-Noise Ratio', fontsize=12, fontweight='bold')
ax5.grid(True, alpha=0.3)

# SFDR
ax6 = fig.add_subplot(gs[1, 2])
sfdr_list = [m['sfdr_db'] for m in measurements]
ax6.plot(Pin_list, sfdr_list, 'o-', ms=8, lw=2, color='purple')
ax6.set_xlabel('Input Power (dBm)', fontsize=11)
ax6.set_ylabel('SFDR (dB)', fontsize=11)
ax6.set_title('Spurious-Free Dynamic Range', fontsize=12, fontweight='bold')
ax6.grid(True, alpha=0.3)

# Phase Histogram
ax7 = fig.add_subplot(gs[2, 0])
ax7.hist(phases_deg, bins=15, edgecolor='k', alpha=0.7, color='purple')
ax7.axvline(phase_mean, color='r', ls='--', lw=2, label=f'μ={phase_mean:.1f}°')
ax7.axvline(phase_mean - phase_std, color='orange', ls=':', lw=1.5, alpha=0.7)
ax7.axvline(phase_mean + phase_std, color='orange', ls=':', lw=1.5, alpha=0.7)
ax7.set_xlabel('Phase (degrees)', fontsize=11)
ax7.set_ylabel('Count', fontsize=11)
ax7.set_title(f'Phase Distribution (σ={phase_std:.1f}°)', fontsize=12, fontweight='bold')
ax7.legend()
ax7.grid(True, alpha=0.3, axis='y')

# Frequency Stability
ax8 = fig.add_subplot(gs[2, 1])
freqs_peak = [m['f_peak']/1e6 for m in measurements]
ax8.plot(Pin_list, freqs_peak, 'o-', ms=8, lw=2, color='orange')
ax8.axhline(np.mean(freqs_peak), color='r', ls='--', lw=2, 
            label=f'μ={np.mean(freqs_peak):.4f} MHz')
ax8.set_xlabel('Input Power (dBm)', fontsize=11)
ax8.set_ylabel('Peak Frequency (MHz)', fontsize=11)
ax8.set_title('Frequency After DDS', fontsize=12, fontweight='bold')
ax8.legend()
ax8.grid(True, alpha=0.3)

# IQ Constellation
ax9 = fig.add_subplot(gs[2, 2])
colors = plt.cm.plasma(np.linspace(0, 1, len(measurements)))
for i, (m, c) in enumerate(zip(measurements[::2], colors[::2])):  # Plot every other
    x_data = np.loadtxt(m['file'], dtype=np.complex128)
    subsample = max(1, len(x_data) // 400)
    label = f"{m['Pin_gen']:+.0f} dBm" if i % 2 == 0 else ""
    ax9.scatter(np.real(x_data[::subsample]), np.imag(x_data[::subsample]),
               alpha=0.3, s=1, color=c, label=label)
ax9.set_xlabel('I (Real)', fontsize=11)
ax9.set_ylabel('Q (Imaginary)', fontsize=11)
ax9.set_title('IQ Constellation', fontsize=12, fontweight='bold')
ax9.legend(fontsize=8, markerscale=5, ncol=2)
ax9.grid(True, alpha=0.3)
ax9.axis('equal')

# Spectrum (middle power)
ax10 = fig.add_subplot(gs[3, :2])
mid_idx = len(measurements) // 2
x_mid = np.loadtxt(measurements[mid_idx]['file'], dtype=np.complex128)
window_mid = np.hanning(len(x_mid))
X_mid = np.fft.fftshift(np.fft.fft(x_mid * window_mid) / len(x_mid))
f_mid = np.fft.fftshift(np.fft.fftfreq(len(x_mid), 1/fs_out))
mag_mid_dbm = 20*np.log10(np.abs(X_mid)/FS) + CAL_CONSTANT_POST
ax10.plot(f_mid/1e6, mag_mid_dbm, lw=1, alpha=0.7)
ax10.axvline(0, color='r', ls='--', lw=2, alpha=0.5, label='DC (after DDS)')
ax10.set_xlabel('Frequency (MHz)', fontsize=11)
ax10.set_ylabel('Power (dBm)', fontsize=11)
ax10.set_title(f'Calibrated Spectrum (Pin={Pin_list[mid_idx]:+.0f} dBm)', 
              fontsize=12, fontweight='bold')
ax10.legend()
ax10.grid(True, alpha=0.3)
ax10.set_xlim([-fs_out/2/1e6, fs_out/2/1e6])

# Summary Table
ax11 = fig.add_subplot(gs[3, 2])
ax11.axis('off')
summary_text = f"""
CALIBRATION SUMMARY

Pre-DDS:  {CAL_CONSTANT_PRE:.2f} dB
Post-DDS: {CAL_CONSTANT_POST:.2f} dB
Net DDS+CIC: {DDS_CIC_NET_EFFECT:.2f} dB

CIC gain: {CIC_GAIN_DB:.2f} dB
Unexplained: {UNACCOUNTED_LOSS:.2f} dB

Accuracy: ±{np.std(errors[:8]):.3f} dB
Phase σ: {phase_std:.2f}°
SNR range: {np.min(snr_list):.1f}-{np.max(snr_list):.1f} dB

Phase-Power corr: {phase_power_corr:.3f}
"""
ax11.text(0.1, 0.5, summary_text, transform=ax11.transAxes,
         fontsize=11, verticalalignment='center', fontfamily='monospace',
         bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plt.savefig('images/ddscic_complete_analysis.pdf', dpi=300, bbox_inches='tight')
plt.show()

# ==========================================
# FINAL REPORT
# ==========================================
print(f"\n{'='*80}")
print("FINAL DIAGNOSTIC REPORT")
print(f"{'='*80}")

print(f" CALIBRATION:")
errors = np.array(Pin_recovered) - np.array(Pin_list)
print(f"  • Linear fit accuracy: ±{np.std(errors[:8]):.3f} dB")
print(f"  • Pin (dBm) ≈ P_out (dBFS) + {CAL_CONSTANT_POST:.2f}")

print(f"\n DDS+CIC CHAIN (QPROD={QPROD}):")
print(f"  • Theoretical CIC gain: {CIC_GAIN_DB:.2f} dB")
print(f"  • Truncation effect: {TRUNC_DB:.2f} dB")
print(f"  • Measured net gain: {DDS_CIC_NET_EFFECT:.2f} dB")
print(f"  • Model mismatch: {residual_loss:.2f} dB")

print(f"\n PHASE:")
if phase_std < 10:
    print(f"   Excellent: {phase_std:.2f}° std dev")
elif phase_std < 20:
    print(f"   Good: {phase_std:.2f}° std dev")
elif abs(phase_power_corr) > 0.7:
    print(f"  ! Phase depends on power (r={phase_power_corr:.2f})")
else:
    print(f"  ! High variation ({phase_std:.2f}°) without power correlation")

## ==========================================
## FULL DSP CHAIN GAIN ANALYSIS
## ==========================================
#print(f"\n{'='*80}")
#print("FULL DSP CHAIN GAIN ANALYSIS")
#print(f"{'='*80}")
#
## PFB scaling
#PFB_GAIN = 2**(-PFB_QOUT)
#PFB_GAIN_DB = 20*np.log10(PFB_GAIN)
#
## CIC gain
#CIC_GAIN_DB = 20*np.log10((R*M)**N)
#
## truncation
#TRUNC_BITS = QPROD_DDS_INTERNAL - CIC_OUTPUT_BITS
#TRUNC_GAIN_DB = -6.02 * TRUNC_BITS
#
## theoretical total
#TOTAL_THEORY_DB = PFB_GAIN_DB + CIC_GAIN_DB + TRUNC_GAIN_DB
#
## measured
#MEASURED_NET_GAIN = CAL_CONSTANT_PRE - CAL_CONSTANT_POST
#
#print("PFB:")
#print(f"  scaling = 2^-{PFB_QOUT}")
#print(f"  gain = {PFB_GAIN_DB:.2f} dB")
#
#print("\nCIC:")
#print(f"  R={R}, M={M}, N={N}")
#print(f"  gain = {CIC_GAIN_DB:.2f} dB")
#
#print("\nDDS+CIC truncation:")
#print(f"  internal width = {QPROD_DDS_INTERNAL} bits")
#print(f"  output width   = {CIC_OUTPUT_BITS} bits")
#print(f"  truncation     = {TRUNC_BITS} bits")
#print(f"  trunc gain     = {TRUNC_GAIN_DB:.2f} dB")
#
#print("\nTOTAL THEORETICAL DSP GAIN:")
#print(f"  G_total = {TOTAL_THEORY_DB:.2f} dB")
#
#print("\nMEASURED GAIN:")
#print(f"  G_measured = {MEASURED_NET_GAIN:.2f} dB")
#
#gain_error = MEASURED_NET_GAIN - TOTAL_THEORY_DB
#
#print("\nGAIN ERROR:")
#print(f"  error = {gain_error:.2f} dB")

## Power Calibration at DDS+CIC Output

From experimental measurements, the relationship between the RF input power and the digital level measured at the DDS+CIC output is:

$$
P_{in}(\text{dBm})=P_{DDS+CIC}(\text{dBFS})+24.28
$$

Where:

$P_{DDS+CIC}$ is the peak amplitude measured in the digital spectrum.

This calibration constant includes:

    * RF front-end attenuation
    * ADC scaling
    * DSP gain
    * DDS+CIC quantization effects

The slope of the calibration fit was:

$$
1.037
$$

which confirms the system behaves linearly over the measured range.